## Setup

In [ ]:
import os
from dotenv import load_dotenv
import numpy as np
import pandas as pd
import re
import xarray as xr
import glob
from tqdm import tqdm
import astropy.units as u
from astropy.coordinates import SkyCoord
import sunpy.coordinates

pd.set_option("future.no_silent_downcasting", True)

In [ ]:
load_dotenv()
RAW_PATH = os.getenv('EVENTS_RAW_PATH')
PROCESSED_PATH = os.getenv('EVENTS_PROCESSED_PATH')
TREATED_PATH = os.getenv('EVENTS_TREATED_PATH')

DATA_YEARS = range(2010, 2025)

## Read Files and create Intermediate CSVs

In [ ]:
def parse_ftp_txt(filepath):
    """Lê arquivos .txt do SWPC com parsing Fixed-Width (FWF) garantindo conformidade de memória sem loops encadeados no Pandas."""
    current_date = None
    skip_lines = 0

    # 1. Identifica cabeçalho e início dos dados
    with open(filepath, 'r') as f:
        for i, line in enumerate(f):
            if line.startswith(':Date:'):
                current_date = line.replace(':Date:', '').strip().replace(' ', '-')
            elif line.startswith('#Event'):
                skip_lines = i + 2
                break

    if not current_date:
        return pd.DataFrame()

    # 2. Definição da geometria das colunas
    colspecs = [
        (0, 10),   # Event       (ex: "3960 +    ")
        (10, 16),  # Begin       (ex: " 0347 ")
        (16, 24),  # Max         (ex: "  0355  ")
        (24, 32),  # End         (ex: "    0359")
        (32, 38),  # Obs         (ex: "  G16 ")
        (38, 41),  # Q           (ex: " 5 ")
        (41, 47),  # Type        (ex: "  XRA ")
        (47, 56),  # Loc_Frq     (ex: " 1-8A     ")
        (56, 74),  # Particulars (ex: "  C2.8    1.2E-03 ")
        (74, 85)   # Reg         (ex: "  3599")
    ]
    cols = ['Event', 'Begin', 'Max', 'End', 'Obs', 'Q', 'Type', 'Loc_Frq', 'Particulars', 'Reg']

    # 3. Leitura do arquivo
    try:
        df_ = pd.read_fwf(
            filepath, skiprows=skip_lines, colspecs=colspecs,
            names=cols, header=None, on_bad_lines='skip'
        )
    except Exception as e:
        print(f"Erro ao ler FWF no arquivo {filepath}: {e}")
        return pd.DataFrame()

    if df_.empty:
        return df_

    # Cria uma cópia profunda garantindo controle in-place seguro
    df_ = df_.copy()

    if 'Date' not in df_.columns:
        df_.insert(0, 'Date', current_date)

    for col in cols:
        df_[col] = df_[col].astype(str).str.strip()

    df_ = df_.replace({
        'nan': np.nan, '////': np.nan, 'None': np.nan, '': np.nan
    }).infer_objects(copy=False)

    df_ = df_[~df_['Event'].str.contains('---', na=False)]

    return df_

In [ ]:
def parse_srs_txt(filepath):
    data = []
    current_date = None
    try:
        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            lines = f.readlines()
    except Exception:
        return pd.DataFrame()

    in_table = False
    for line in lines:
        # Troca espaços inquebráveis por espaços normais e limpa as pontas
        line_clean = line.replace('\xa0', ' ').strip()

        if line_clean.startswith(':Issued:'):
            date_str = line_clean.replace(':Issued:', '').replace('UTC', '').strip()
            try:
                current_date = pd.to_datetime(date_str).strftime('%Y-%m-%d')
            except:
                pass

        # GATILHO BLINDADO: ignora a quantidade de espaços no meio da frase
        if 'Regions with Sunspots' in line_clean and line_clean.startswith('I.'):
            in_table = True
            continue

        if in_table and (line_clean.startswith('IA.') or line_clean.startswith('II.')):
            break

        if in_table:
            parts = line_clean.split()
            if len(parts) >= 8 and parts[0].isdigit():
                nmbr_to_nn = parts[:7]
                mag_type = " ".join(parts[7:])
                data.append([current_date] + nmbr_to_nn + [mag_type])

    cols = ['Date', 'Nmbr', 'Location', 'Lo', 'Area', 'Z', 'LL', 'NN', 'Mag_Type']
    return pd.DataFrame(data, columns=cols)

In [ ]:
for y_ in tqdm(DATA_YEARS, desc="Processando Anos"):
    y_str = str(y_)
    output_dir = os.path.join(PROCESSED_PATH, y_str)
    os.makedirs(output_dir, exist_ok=True)

    # 1. Events
    events_files = glob.glob(os.path.join(RAW_PATH, y_str, '*', 'sci_xrsf-l2-flsum_*.nc'))
    df_events_list = []
    for f in events_files:
        try:
            ds = xr.open_dataset(f)
            df = ds.to_dataframe().reset_index()
            df_events_list.append(df)
            ds.close()
        except Exception as e:
            print(f"Erro ao ler NC (Events): {f} - {e}")
    if df_events_list:
        pd.concat(df_events_list, ignore_index=True).to_csv(os.path.join(output_dir, f'events_{y_str}.csv'), index=False)

    # 2. FTP
    swpc_files = glob.glob(os.path.join(RAW_PATH, y_str, f'{y_str}_SWPC_events', f'{y_str}_events', '*events.txt'))
    df_swpc_list = [parse_ftp_txt(f) for f in swpc_files]
    df_swpc_list = [df for df in df_swpc_list if not df.empty]
    if df_swpc_list:
        pd.concat(df_swpc_list, ignore_index=True).to_csv(os.path.join(output_dir, f'FTP_{y_str}.csv'), index=False)

    # 3. SSW
    ssw_files = glob.glob(os.path.join(RAW_PATH, y_str, 'SSW', '*', 'SSW_*.csv'))
    df_ssw_list = [pd.read_csv(f) for f in ssw_files]
    if df_ssw_list:
        pd.concat(df_ssw_list, ignore_index=True).to_csv(os.path.join(output_dir, f'SSW_{y_str}.csv'), index=False)

    # 4. Locations
    if y_ >= 2017:
        locations_files = glob.glob(os.path.join(RAW_PATH, y_str, 'NCEI_FLLOC', '*', 'sci_xrsf-l2-flloc_*.nc'))
        df_loc_list = []
        for f in locations_files:
            try:
                ds = xr.open_dataset(f)
                df = ds.to_dataframe().reset_index()
                df_loc_list.append(df)
                ds.close()
            except Exception as e:
                print(f"Erro ao ler NC (Locations): {f} - {e}")
        if df_loc_list:
            pd.concat(df_loc_list, ignore_index=True).to_csv(os.path.join(output_dir, f'Locations_{y_str}.csv'), index=False)

    # 5. Reports
    search_pattern = os.path.join(RAW_PATH, y_str, '**', '*SRS*')
    srs_files = [f for f in glob.glob(search_pattern, recursive=True) if os.path.isfile(f)]

    df_srs_list = [parse_srs_txt(f) for f in srs_files]
    df_srs_list = [df for df in df_srs_list if not df.empty]
    if df_srs_list:
        pd.concat(df_srs_list, ignore_index=True).to_csv(os.path.join(output_dir, f'SRS_{y_str}.csv'), index=False)

## Read Intermediate CSVs

In [ ]:
PROCESSED_DATA = {}
DATASETS = ['events', 'FTP', 'SSW', 'Locations', 'SRS']

for year in tqdm(range(2010, 2025), desc="Carregando CSVs Processados"):
    y_str = str(year)
    PROCESSED_DATA[y_str] = {}

    for dataset in DATASETS:
        if dataset == 'Locations' and year < 2017:
            PROCESSED_DATA[y_str][dataset] = pd.DataFrame()
            continue

        file_path = os.path.join(PROCESSED_PATH, y_str, f'{dataset}_{y_str}.csv')

        if os.path.exists(file_path):
            try:
                # Condicional de tipagem para proteger a matemática futura:
                if dataset in ['FTP', 'SSW']:
                    PROCESSED_DATA[y_str][dataset] = pd.read_csv(file_path, dtype=str)
                else:
                    PROCESSED_DATA[y_str][dataset] = pd.read_csv(file_path) # Deixa o Pandas inferir os floats para Science e Locations
            except pd.errors.EmptyDataError:
                PROCESSED_DATA[y_str][dataset] = pd.DataFrame()
        else:
            PROCESSED_DATA[y_str][dataset] = pd.DataFrame()

## Preparação, Padronização e Correção das Bases

In [ ]:
# =============================================================================
# 1.1 UNIFICAÇÃO TEMPORAL DOS DADOS (Carga na Memória)
# =============================================================================
# Os dados brutos foram previamente extraídos, processados e armazenados em fatias anuais.
df_sci_list, df_ftp_list, df_ssw_list, df_srs_list = [], [], [], []

for y in DATA_YEARS:
    y_str = str(y)
    if not PROCESSED_DATA[y_str]['events'].empty:
        df_sci_list.append(PROCESSED_DATA[y_str]['events'])
    if not PROCESSED_DATA[y_str]['FTP'].empty:
        df_ftp_list.append(PROCESSED_DATA[y_str]['FTP'])
    if not PROCESSED_DATA[y_str]['SSW'].empty:
        df_ssw_list.append(PROCESSED_DATA[y_str]['SSW'])
    if not PROCESSED_DATA[y_str]['SRS'].empty:
        df_srs_list.append(PROCESSED_DATA[y_str]['SRS'])

df_sci_raw = pd.concat(df_sci_list, ignore_index=True) if df_sci_list else pd.DataFrame()
df_ftp_raw = pd.concat(df_ftp_list, ignore_index=True) if df_ftp_list else pd.DataFrame()
df_ssw_raw = pd.concat(df_ssw_list, ignore_index=True) if df_ssw_list else pd.DataFrame()

df_srs_raw = pd.concat(df_srs_list, ignore_index=True) if df_srs_list else pd.DataFrame()
if not df_srs_raw.empty:
    df_srs_raw['Date'] = pd.to_datetime(df_srs_raw['Date'], errors='coerce').dt.date

In [ ]:
# =============================================================================
# 1.2 PADRONIZAÇÃO DE ESCALAS, TEMPO E METADADOS
# =============================================================================

def goes_class_to_log10(flare_class):
    if pd.isna(flare_class) or not isinstance(flare_class, str):
        return np.nan

    # Busca um caractere de A a X seguido por números (com ou sem ponto decimal)
    match = re.search(r'([A-X])(\d+\.?\d*)', flare_class.upper())
    if not match:
        return np.nan

    letter = match.group(1)
    scale = {'A': -8, 'B': -7, 'C': -6, 'M': -5, 'X': -4}

    if letter not in scale:
        return np.nan

    try:
        multiplier = float(match.group(2))
        if multiplier <= 0:
            return np.nan
        return np.log10(multiplier) + scale[letter]
    except ValueError:
        return np.nan

In [ ]:
# --- EVENTS ---
df_filtered = df_sci_raw[df_sci_raw['status'] == 'EVENT_PEAK']

df_sci = pd.DataFrame({
    'flare_id': df_filtered['flare_id'].astype(str).str.replace(r'\.0$', '', regex=True),
    'peak_time': pd.to_datetime(df_filtered['time'], utc=True),
    'flare_class': df_filtered['flare_class'],
    'log10_intensity': np.log10(df_filtered['xrsb_flux'])
})

In [ ]:
# --- SSW ---
df_ssw = df_ssw_raw.copy()

df_ssw.columns = [str(col).strip() for col in df_ssw.columns]

if not df_ssw.empty and 'Start' in df_ssw.columns and 'Peak' in df_ssw.columns:
    df_ssw['date_str'] = df_ssw['Start'].str[:10]
    df_ssw['peak_time'] = pd.to_datetime(df_ssw['date_str'] + ' ' + df_ssw['Peak'], utc=True)
    df_ssw['log10_intensity'] = df_ssw['GOES Class'].apply(goes_class_to_log10)
    df_ssw['Reg'] = df_ssw['Derived Position'].str.extract(r'\(\s*(\d+)\s*\)')[0]

    # Filtra mantendo apenas as colunas úteis
    df_ssw = df_ssw[['EName', 'peak_time', 'GOES Class', 'log10_intensity', 'Reg']].dropna(subset=['peak_time'])
else:
    df_ssw = pd.DataFrame(columns=['EName', 'peak_time', 'GOES Class', 'log10_intensity', 'Reg'])

In [ ]:
# --- FTP ---
valid_ftp_cols = [str(col).strip() for col in df_ftp_raw.columns]

if not df_ftp_raw.empty and 'Date' in valid_ftp_cols and 'Max' in valid_ftp_cols:

    df_temp = df_ftp_raw.rename(columns=dict(zip(df_ftp_raw.columns, valid_ftp_cols)))

    # FILTRO: Mantém apenas as explosões de raios-X (XRA)
    if 'Type' in df_temp.columns:
        df_temp = df_temp[df_temp['Type'] == 'XRA'].copy()

    max_time_series = df_temp['Max'].astype(str).str.zfill(4).apply(
        lambda val: f"{val[:2]}:{val[2:]}:00" if val.isdigit() else np.nan
    )

    df_ftp = pd.DataFrame({
        'Event': df_temp['Event'],
        'peak_time': pd.to_datetime(df_temp['Date'] + ' ' + max_time_series, errors='coerce', utc=True),
        'Particulars': df_temp['Particulars'],
        'log10_intensity': df_temp['Particulars'].apply(goes_class_to_log10),
        'Reg': df_temp['Reg'] if 'Reg' in df_temp.columns else np.nan
    }).dropna(subset=['peak_time'])

else:
    df_ftp = pd.DataFrame(columns=['Event', 'peak_time', 'Particulars', 'log10_intensity', 'Reg'])

In [ ]:
# =============================================================================
# 1.3 CORREÇÃO DE ESCALA SWPC E SSW
# =============================================================================
cutoff_date = pd.to_datetime('2019-12-09', utc=True)
exact_correction = np.abs(np.log10(0.7))
mask_ssw = df_ssw['peak_time'] < cutoff_date
df_ssw.loc[mask_ssw, 'log10_intensity'] += exact_correction

if not df_ftp.empty:
    mask_ftp = df_ftp['peak_time'] < cutoff_date
    df_ftp.loc[mask_ftp, 'log10_intensity'] += exact_correction

In [ ]:
print(f"🚀 Tratamento Inicial Concluído:")
print(f"Science-Quality: {len(df_sci)} eventos de pico prontos.")
print(f"SSW: {len(df_ssw)} eventos padronizados e escala corrigida.")
print(f"FTP: {len(df_ftp)} eventos padronizados e escala corrigida.")

## Algoritmo de Matching (Science-Quality <-> Operacionais)

Para cada flare na base Science-Quality, buscamos a correspondência nos catálogos operacionais (FTP e SSW).

In [ ]:
def find_match(target_row, df_catalogue, time_tol_min=15, flux_tol_log=0.3):
    if df_catalogue.empty or pd.isna(target_row['peak_time']) or pd.isna(target_row['log10_intensity']):
        return None

    time_diffs = (df_catalogue['peak_time'] - target_row['peak_time']).abs().dt.total_seconds() / 60.0
    flux_diffs = (df_catalogue['log10_intensity'] - target_row['log10_intensity']).abs()

    mask = (time_diffs <= time_tol_min) & (flux_diffs <= flux_tol_log)
    candidates = df_catalogue[mask].copy()

    if candidates.empty:
        return None

    candidates['time_diff'] = time_diffs[mask]
    best_match = candidates.sort_values(by='time_diff').iloc[0]

    return best_match['Reg']

In [ ]:
df_sci['ftp_AR'] = np.nan
df_sci['ssw_AR'] = np.nan

print("Iniciando Matching (Isso pode levar alguns segundos)...")

for idx, row in tqdm(df_sci.iterrows(), total=len(df_sci), desc="Cruzando Catálogos"):
    df_sci.at[idx, 'ftp_AR'] = find_match(row, df_ftp)
    df_sci.at[idx, 'ssw_AR'] = find_match(row, df_ssw)

print("\n✅ Matching Inicial Concluído.")

## Preparação e Pivotamento Para o Augmentation Geométrico

In [ ]:
# 3.1 Unificando os DataFrames de Locations
df_loc_list = []
for y in range(2017, 2025):
    y_str = str(y)
    if not PROCESSED_DATA[y_str]['Locations'].empty:
        df_loc_list.append(PROCESSED_DATA[y_str]['Locations'])

df_loc_raw = pd.concat(df_loc_list, ignore_index=True) if df_loc_list else pd.DataFrame()

In [ ]:
# 3.2 Pivotamento
if not df_loc_raw.empty:
    # Padroniza o flare_id para string removendo ".0", garantindo o Left Join futuro
    df_loc_raw['flare_id'] = df_loc_raw['flare_id'].astype(str).str.replace(r'\.0$', '', regex=True)

    df_loc_pivot = df_loc_raw.pivot_table(
        index='flare_id',
        columns='coordinate',
        values='flloc_xy',
        aggfunc='first'
    ).reset_index()

    df_loc_pivot = df_loc_pivot.rename(columns={
        0.0: 'hpc_x', 1.0: 'hpc_y',
        0: 'hpc_x', 1: 'hpc_y',
        '0': 'hpc_x', '1': 'hpc_y',
        '0.0': 'hpc_x', '1.0': 'hpc_y'
    })
else:
    df_loc_pivot = pd.DataFrame(columns=['flare_id', 'hpc_x', 'hpc_y'])

In [ ]:
# 3.3 Filtro do Sub-DataFrame
geom_start_date = pd.to_datetime('2017-02-09', utc=True)

df_sci['ftp_AR'] = pd.to_numeric(df_sci['ftp_AR'], errors='coerce')
mask_needs_geom = (df_sci['ftp_AR'].isna() | (df_sci['ftp_AR'] == 0)) & (df_sci['peak_time'] >= geom_start_date)
df_sci_unmatched = df_sci[mask_needs_geom].copy()

In [ ]:
# 3.4 Merge (Left Join)
df_sci_geom = pd.merge(
    df_sci_unmatched,
    df_loc_pivot[['flare_id', 'hpc_x', 'hpc_y']],
    on='flare_id',
    how='left'
)

df_sci_geom = df_sci_geom.dropna(subset=['hpc_x', 'hpc_y'])

print(f"🚀 Preparações Concluídas:")
print(f"Flares sem AR elegíveis para resgate espacial (>= 2017): {len(df_sci_unmatched)}")
print(f"Flares onde o pivotamento encontrou coordenadas XY: {len(df_sci_geom)}")

## Cálculo de Distância Euclidiana (SRS)

In [ ]:
def parse_srs_location(loc_str):
    if not isinstance(loc_str, str) or len(loc_str) < 6:
        return np.nan, np.nan
    lat_dir, lat_val = loc_str[0], loc_str[1:3]
    lon_dir, lon_val = loc_str[3], loc_str[4:6]

    try:
        lat = float(lat_val) * (1 if lat_dir == 'N' else -1)
        lon = float(lon_val) * (1 if lon_dir == 'E' else -1)
        return lat, lon
    except ValueError:
        return np.nan, np.nan

def stonyhurst_to_hpc(lat, lon, obstime):
    try:
        coord = SkyCoord(lon * u.deg, lat * u.deg,
                         frame="heliographic_stonyhurst",
                         obstime=obstime)
        hpc_coord = coord.transform_to(sunpy.coordinates.Helioprojective(obstime=obstime))
        return hpc_coord.Tx.value, hpc_coord.Ty.value
    except Exception:
        return np.nan, np.nan

In [ ]:
df_sci_geom['nearest_AR_srs'] = np.nan
df_sci_geom['min_distance_arcsec'] = np.nan

print("Iniciando Cálculo de Distâncias Geométricas...")

for idx, flare in tqdm(df_sci_geom.iterrows(), total=len(df_sci_geom), desc="Calculando HPC e Distâncias"):
    flare_time = flare['peak_time']
    flare_date = flare_time.date()
    flare_hpc_x = flare['hpc_x']
    flare_hpc_y = flare['hpc_y']

    active_ars_today = df_srs_raw[df_srs_raw['Date'] == flare_date]

    if active_ars_today.empty:
        continue

    min_dist = np.inf
    best_ar = np.nan

    for _, ar_row in active_ars_today.iterrows():
        lat, lon = parse_srs_location(ar_row['Location'])
        if pd.isna(lat) or pd.isna(lon):
            continue
        ar_hpc_x, ar_hpc_y = stonyhurst_to_hpc(lat, lon, flare_time)
        if pd.isna(ar_hpc_x) or pd.isna(ar_hpc_y):
            continue

        distance = np.sqrt((ar_hpc_x - flare_hpc_x)**2 + (ar_hpc_y - flare_hpc_y)**2)
        if distance < min_dist:
            min_dist = distance
            best_ar = ar_row['Nmbr']

    if min_dist != np.inf:
        df_sci_geom.at[idx, 'nearest_AR_srs'] = best_ar
        df_sci_geom.at[idx, 'min_distance_arcsec'] = min_dist

print("\n✅ Cálculo Geométrico Concluído!")

## Atribuição Final por Geometria

In [ ]:
threshold_arcsec = 250.0
df_sci_geom['rescued_AR'] = np.where(
    df_sci_geom['min_distance_arcsec'] < threshold_arcsec,
    df_sci_geom['nearest_AR_srs'],
    np.nan
)

rescued_count = df_sci_geom['rescued_AR'].notna().sum()
print(f"Flares resgatados com sucesso via geometria (distância < {threshold_arcsec} arcsec): {rescued_count}")

In [ ]:
df_sci['geom_AR'] = np.nan
map_rescued_ar = df_sci_geom.dropna(subset=['rescued_AR']).set_index('flare_id')['rescued_AR']
mask_to_update = df_sci['flare_id'].isin(map_rescued_ar.index)

df_sci.loc[mask_to_update, 'geom_AR'] = df_sci.loc[mask_to_update, 'flare_id'].map(map_rescued_ar)

In [ ]:
# TRATAMENTO DE INVÁLIDOS: Transformar os zeros em nulos (NaN) para permitir o resgate
df_sci['ftp_AR'] = df_sci['ftp_AR'].replace(0, np.nan)
df_sci['ssw_AR'] = df_sci['ssw_AR'].replace(0, np.nan)

# CONSOLIDAÇÃO (Hierarquia: FTP -> Geometria -> SSW)
df_sci['matched_AR'] = df_sci['ftp_AR'].fillna(df_sci['geom_AR']).fillna(df_sci['ssw_AR'])

# RASTREABILIDADE DA FONTE
df_sci['match_source'] = np.nan
df_sci.loc[df_sci['matched_AR'] == df_sci['ftp_AR'], 'match_source'] = 'FTP'
df_sci.loc[(df_sci['matched_AR'] == df_sci['geom_AR']) & df_sci['match_source'].isna(), 'match_source'] = 'Geometry_SRS'
df_sci.loc[(df_sci['matched_AR'] == df_sci['ssw_AR']) & df_sci['match_source'].isna(), 'match_source'] = 'SSW'

# MÉTRICAS FINAIS
total_sci = len(df_sci)
final_matched = df_sci['matched_AR'].notna().sum()

In [ ]:
print("\n--- Resumo Final de Atribuição de ARs ---")
print(f"Total de flares Science-Quality: {total_sci}")
print(f"Total com AR atribuída: {final_matched} ({(final_matched / total_sci * 100):.1f}%)")
print(f"Total mantidos como 'Sem AR': {total_sci - final_matched}")

## Consolidação e Exportação (Treated/Clean Layer)

In [ ]:
df_sci['match_source'] = df_sci['match_source'].fillna('UNMATCHED')
df_sci['matched_AR'] = pd.to_numeric(df_sci['matched_AR'], errors='coerce').astype('Int64')
df_sci = df_sci.drop(columns=['ftp_AR', 'ssw_AR', 'geom_AR'], errors='ignore')

In [ ]:
df_final = df_sci.sort_values(by='peak_time').reset_index(drop=True)
df_final = df_final.rename(columns={
    'matched_AR': 'active_region_no',
    'match_source': 'ar_provenance'
})

In [ ]:
os.makedirs(TREATED_PATH, exist_ok=True)
output_filepath = os.path.join(TREATED_PATH, 'treated_events.parquet')

df_final.to_parquet(output_filepath, index=False)

In [ ]:
print("🚀 Pipeline de Tratamento Finalizado com Sucesso!")
print(f"Dataset exportado para: {output_filepath}")
print(f"Dimensões finais da matriz: {df_final.shape}")
print("\nAmostra dos dados finais:")
display(df_final.head())